In [17]:
from pyspark import SparkContext

In [18]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
from pyspark.sql.types import Row
from pyspark.sql import SQLContext

In [19]:
sc = SparkContext(master='local', appName='UDFyreplicacion')

ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=UDFyreplicacion, master=local) created by __init__ at /tmp/ipykernel_92/1368799085.py:1 

In [27]:
sqlContext = SQLContext(sc)

/usr/local/lib/python3.10/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [28]:
path = "/home/jovyan/data/"

In [29]:
!head -n 5 /home/jovyan/data/deportistaError.csv

deportista_id,nombre,genero,edad,altura,peso,equipo_id
1,A Dijiang,1,24,180,80,199
2,A Lamusi,1,23,170,60,199
3,Gunnar Nielsen Aaby,1,24,,,273
4,Edgar Lindenau Aabye,1,34,,,278


In [30]:
deportistaError = sc.textFile(path + "deportistaError.csv") \
    .map(lambda l : l.split(","))

In [31]:
def eliminarEncabezados(indice, iterador):
    return iter(list(iterador)[1:])

In [32]:
deportistaError = deportistaError.mapPartitionsWithIndex(eliminarEncabezados)

In [33]:
deportistaError.take(2)

[['1', 'A Dijiang', '1', '24', '180', '80', '199'],
 ['2', 'A Lamusi', '1', '23', '170', '60', '199']]

In [34]:
deportistaError.map(lambda l: (
    l[0],
    l[1],
    l[2],
    l[3],
    l[4],
    l[5],
    l[6]
))
schema = StructType([
    StructField("deportista_id", StringType(), False),
    StructField("nombre", StringType(), False),
    StructField("genero", StringType(), False),
    StructField("edad", StringType(), False),
    StructField("altura", StringType(), False),
    StructField("peso", StringType(), False),
    StructField("equipo_id", StringType(), False)
])
deportistaErrorDF = sqlContext.createDataFrame(deportistaError, schema)

In [35]:
deportistaErrorDF.show()

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24|   180|  80|      199|
|            2|            A Lamusi|     1|  23|   170|  60|      199|
|            3| Gunnar Nielsen Aaby|     1|  24|      |    |      273|
|            4|Edgar Lindenau Aabye|     1|  34|      |    |      278|
|            5|Christine Jacoba ...|     2|  21|   185|  82|      705|
|            6|     Per Knut Aaland|     1|  31|   188|  75|     1096|
|            7|        John Aalberg|     1|  31|   183|  72|     1096|
|            8|"Cornelia ""Cor""...|     2|  18|   168|    |      705|
|            9|    Antti Sami Aalto|     1|  26|   186|  96|      350|
|           10|"Einar Ferdinand ...|     1|  26|      |    |      350|
|           11|  Jorma Ilmari Aalto|     1|  22|   182|76.5|      350|
|     

In [39]:
from pyspark.sql.functions import udf
def conversionEnteros(valor):
    return int(valor) if len(valor) > 0 else None
conversionEnteros_udf = udf(lambda z: conversionEnteros(z), IntegerType())
sqlContext.udf.register("conversionEnteros_udf", conversionEnteros_udf)

In [41]:
deportistaErrorDF.select(conversionEnteros_udf("altura")\
                        .alias("alturaUDF")).show()

+---------+
|alturaUDF|
+---------+
|      180|
|      170|
|     NULL|
|     NULL|
|      185|
|      188|
|      183|
|      168|
|      186|
|     NULL|
|      182|
|      172|
|      159|
|      171|
|     NULL|
|      184|
|      175|
|      189|
|     NULL|
|      176|
+---------+
only showing top 20 rows


In [42]:
from pyspark.storagelevel import StorageLevel

In [43]:
deportistaErrorDF.is_cached

False

In [44]:
deportistaErrorDF.rdd.chache()

AttributeError: 'RDD' object has no attribute 'chache'